<a href="https://colab.research.google.com/github/micko112/LLM-Chatbot---Rent-Rent-Out/blob/main/LLM_Chatbot.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q langchain langchain-openai langchain-chroma tiktoken langchain-community


In [ ]:
import os
from google.colab import userdata
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_chroma import Chroma
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

In [ ]:
os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_API_KEY')

In [ ]:
platform_rules = """
1. Pravo na postavljanje oglasa imaju isključivo lica koja su punoletna (18+ godina) i koja su prošla email verifikaciju.
2. Korisnici koji su prošli identifikaciju (KYC — upload ličnog dokumenta) dobijaju oznaku "Verifikovan" na profilu i imaju veće poverenje među drugim korisnicima.
3. Postavljanje oglasa je besplatno. Promotivne opcije se naplaćuju u kreditima: FEATURED 500 RSD (7 dana, vrh pretrage), PRIORITY 250 RSD (3 dana), HIGHLIGHTED 100 RSD (30 dana, vizuelni badge).
4. Dopuna kredita vrši se uplatom na zvanični račun platforme (Dimitrije Mitic, 265-0000006785327-58) uz obavezno korišćenje jedinstvenog poziva na broj koji se generiše na stranici `/credit`.
5. Krediti nisu povratni nakon što su iskorišćeni za aktivaciju promocije. Ako promocija nije konzumirana, korisnik može tražiti povraćaj u roku od 7 dana na mejl podrške.
6. Depozit koji zakupac eventualno uplaćuje zakupodavcu obavezno se vraća po isteku ugovora, ukoliko predmet iznajmljivanja nije oštećen i ukoliko je vraćen u dogovorenom roku.
7. Stanje iznajmljene stvari (fotografije, funkcionalnost) obavezno se dokumentuje pri primopredaji — i preporučuje se razmena tih fotografija u chat-u platforme kao dokaz.
8. Ukoliko stanodavac (vlasnik stvari) otkaže već prihvaćen ugovor bez opravdanog razloga, zakupac ima pravo na prioritetnu podršku i eventualnu naknadu troškova dokazanih kroz chat istoriju.
9. Otkazni rok za već prihvaćen ugovor je minimum 48 sati pre datuma početka — kasnije otkazivanje može rezultirati negativnom recenzijom i suspenzijom naloga kod ponavljanja.
10. Oštećenje iznajmljene stvari mora biti prijavljeno u roku od 48 sati od primopredaje. Prijava se vrši kroz chat platforme i mejlom na izdajemiznajmljujem.rs@gmail.com sa fotografijama.
11. Platforma ne posreduje u plaćanju naknade za rental (iznajmljivanje) — to se dogovara direktno između korisnika. Platforma naplaćuje isključivo promotivne opcije putem sistema kredita.
12. Zabranjeno je oglašavanje: oružja, narkotika, životinja, ljudskih organa, krađene robe, falsifikata, pirotehnike, alkohola licima mlađim od 18 godina, kao i bilo kakvih nelegalnih usluga.
13. Korisnik ima pravo na brisanje naloga i svih ličnih podataka u skladu sa Zakonom o zaštiti podataka o ličnosti (ZZPL) i GDPR-om — zahtev se šalje na izdajemiznajmljujem.rs@gmail.com.
14. Kasno vraćanje iznajmljene stvari može rezultirati dodatnom naknadom za svaki započet dan kašnjenja prema ceni iz originalnog oglasa.
15. Sporovi oko ugovora rešavaju se najpre direktno kroz chat platforme. Ukoliko korisnici ne postignu dogovor, slučaj se eskaliraju na izdajemiznajmljujem.rs@gmail.com — admini posreduju u roku od 3 radna dana.
16. Prijava sumnje na prevaru (traženje uplate na privatne račune van /credit stranice, lažni oglasi, pretnje) mora biti bezuslovna — suspenzija naloga je trenutna, a slučaj se predaje nadležnim organima ako postoje elementi krivičnog dela.
17. Recenzije se mogu ostaviti isključivo nakon završenog ugovora, u roku od 30 dana. Lažne recenzije, međusobno "nameštanje" i vređanja se brišu bez upozorenja.
18. Prava potrošača u skladu sa Zakonom o zaštiti potrošača — pravo na reklamaciju neispravne usluge, pravo na informisanost, pravo na zaštitu od nepoštene poslovne prakse — primenjuju se na sve komercijalne transakcije preko platforme.
"""

In [ ]:
with open("pravila_izdavanja.txt", "w", encoding = "utf-8") as f:
  f.write(platform_rules)

In [ ]:
loader = TextLoader("pravila_izdavanja.txt")
document = loader.load()
print("Successfuly loaded file")

Successfuly loaded file


CHUNKING

In [ ]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 150,
    chunk_overlap = 20
)
splitted_docs = text_splitter.split_documents(document)
print(f"Splitted on {len(splitted_docs)} chunks")

Splitted on 34 chunks


SBERT embedding




In [ ]:
embeddings_model = OpenAIEmbeddings(model="text-embedding-3-small")

vectors = Chroma.from_documents(
    documents = splitted_docs,
    embedding=embeddings_model
)

In [ ]:
retriever = vectors.as_retriever(search_kwargs={"k": 2})

In [ ]:
llm = ChatOpenAI(model = "gpt-4o-mini", temperature=0.6)

prompt_template = PromptTemplate(
    template = """Ti si gotivni ljubazni dobri agent za korisničku podršku sajta 'Izdajem Iznajmljujem'.
Na osnovu ponuđenog KONTEKSTA, odgovori na PITANJE korisnika.
Ako u KONTEKSTU nema odgovora, reci: "Nažalost, nemam tu informaciju u pravilniku." Nemoj nikada da izmišljaš pravila.

KONTEKST:
{context}

PITANJE KORISNIKA: {question}

ODGOVOR:""",
    input_variables=["context", "question"]
)

In [ ]:
def format_docs(docs):
  return "\n\n".join(doc.page_content for doc in docs)

rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt_template
    | llm
    | StrOutputParser()

)

In [ ]:
from typing import TypedDict
from langgraph.graph import StateGraph, START, END

In [ ]:
# question = "Da li se placa  za oglas na vašem sajtu?"
# print(f"Korisnik pita: {question}")

# answer = rag_chain.invoke(question)
# print(f"Bot odgovara:\n{answer}")

In [ ]:
class AgentState(TypedDict):
    question: str
    answer: str

In [ ]:
def state_changer(state):
  question = state["question"]

  prompt = f"""Da li se ovo pitanje odnosi na pravila, oglase, depozite, plaćanje, kredite ili funkcionisanje sajta za nekretnine?
    Pitanje: {question}
    Odgovori ISKLJUČIVO sa DA ili NE."""

  decision = llm.invoke(prompt).content.strip().lower()

  if "da" in decision:
    return "rag_node"
  else:
    return "chat_node"

In [ ]:
def rag_node(state):
  answer = rag_chain.invoke(state["question"])
  return {"answer": answer}

In [ ]:
def chat_node(state):
  question = state["question"]
  prompt = f"""Ti si gotivan ljubazan i dobar asistent na sajtu Izdajem Iznajmljujem.
  Korisnik ti kaže: '{question}'. Odgovori kratko i prijateljski."""
  answer = llm.invoke(prompt).content
  return {"answer": answer}

In [ ]:
workflow = StateGraph(AgentState)

workflow.add_node("rag_node", rag_node)
workflow.add_node("chat_node", chat_node)

workflow.add_conditional_edges(
    START,
    state_changer,
    {
        "rag_node": "rag_node",
        "chat_node": "chat_node"
    }
)


In [ ]:
workflow.add_edge("rag_node", END)
workflow.add_edge("chat_node", END)

In [ ]:
agent_app = workflow.compile()

In [ ]:
question_1 = "Da li se placa za oglas na vašem sajtu?"
print(f"KORISNIK: {question_1}")
result_1 = agent_app.invoke({"question": question_1})
print(f"BOT: {result_1['answer']}\n")
print("-" * 50)

# Test 2: Pitanje za običan razgovor (Testiramo skretničara!)
question_2 = "Da li znas kvadratnu jednacinu?"
print(f"\nKORISNIK: {question_2}")
result_2 = agent_app.invoke({"question": question_2})
print(f"BOT: {result_2['answer']}")

KORISNIK: Da li se placa za oglas na vašem sajtu?
BOT: Postavljanje oglasa na našem sajtu je besplatno. Međutim, promotivne opcije kao što su FEATURED i PRIORITY se naplaćuju u kreditima. FEATURED košta 500 RSD za 7 dana, dok PRIORITY košta 250 RSD za 3 dana.

--------------------------------------------------

KORISNIK: Da li znas kvadratnu jednacinu?
BOT: Naravno! Kvadratna jednačina je oblik \( ax^2 + bx + c = 0 \), gde su \( a \), \( b \) i \( c \) koeficijenti. Ako trebaš pomoć oko rešenja, slobodno pitaj! 😊
